In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_rel
import warnings
warnings.filterwarnings("ignore")
from scipy.stats import skew
from scipy.stats import kurtosis
from scipy.stats import wilcoxon

# Test de hipótesis para óxidos de cobre segmentado por control de calidad

## 1) Test de hipótesis para extracción de cobre mediante tierras

### 1.1) Test de hipotesis normalidad para óxidos de cobre metodo de extraccion por tierras

In [4]:
# 1. Definición de columnas y carga de datos
# En las columnas deseadas, Zona Min como nueva columna para ser utilizada como filtro
columnas_deseadas = ['Zona Min', 'ext_cut_rip_ph', 'ext_cut_rip_selec_col']
excelPath = r"C:\Users\Diego\Desktop\Ultimo Esfuerzo\Copia de Base de Datos Cuprochlor Sin T Final.xlsx"

df3 = pd.read_excel(excelPath, "BD elim y reducido (3)", usecols=columnas_deseadas)

# 2. Cálculo de la variable diferencia
df3['diferencia'] = df3['ext_cut_rip_ph'] - df3['ext_cut_rip_selec_col']

# 3. Filtrado por zona mineralógica (Óxidos de Cobre - CUO)
# Limpiamos valores nulos específicos de este segmento
datos_oxidos = df3[df3['Zona Min'] == 'CUO']['diferencia'].dropna()

# 4.Calculo de asimetría 
datos=datos_oxidos
asimetria=skew(datos)
print("Asimetria:",asimetria)

# 5.Calculo de Curtosis
datos=datos_oxidos
curtosis=kurtosis(datos)
print("Curtosis:",curtosis)

# 6. Realizar el Test de Shapiro-Wilk
# Nota: Este test no requiere parámetros adicionales (mu, sigma)
shapiro_stat, p_value = stats.shapiro(datos_oxidos)

# 7. Mostrar e interpretar resultados
print("--- TEST DE NORMALIDAD (SHAPIRO-WILK) - ZONA: CUO (ÓXIDOS) ---")
print(f"Estadístico W: {shapiro_stat:.4f}")
print(f"P-valor: {p_value:.4f}")

print("\nInterpretación:")
alpha = 0.05
if p_value > alpha:
    print("Resultado: Los datos de la zona CUO parecen seguir una distribución NORMAL.")
    print("Interpretación: No se rechaza la hipótesis nula (p > 0.05).")
else:
    print("Resultado: Los datos de la zona CUO NO siguen una distribución normal.")
    print("Interpretación: Se rechaza la hipótesis nula (p < 0.05).")

print(f"\nCantidad de muestras analizadas en CUO: {len(datos_oxidos)}")

Asimetria: -0.7092850246236427
Curtosis: -0.18955671122830964
--- TEST DE NORMALIDAD (SHAPIRO-WILK) - ZONA: CUO (ÓXIDOS) ---
Estadístico W: 0.9285
P-valor: 0.3255

Interpretación:
Resultado: Los datos de la zona CUO parecen seguir una distribución NORMAL.
Interpretación: No se rechaza la hipótesis nula (p > 0.05).

Cantidad de muestras analizadas en CUO: 13


### 1.2) Test t Student para óxidos de cobre metodo de extraccion por tierras

In [8]:
# 1. Definición de columnas y carga de datos
columnas_deseadas = ['Zona Min', 'ext_cut_rip_ph', 'ext_cut_rip_selec_col']
excelPath = r"C:\Users\Diego\Desktop\Ultimo Esfuerzo\Copia de Base de Datos Cuprochlor Sin T Final.xlsx"

df3 = pd.read_excel(excelPath, "BD elim y reducido (3)", usecols=columnas_deseadas)

# 2. Filtrado por zona (CUO - Óxidos de Cobre) y limpieza de nulos
# Filtramos primero la zona y luego eliminamos filas donde falte alguno de los dos datos
df_oxidos = df3[df3['Zona Min'] == 'CUO'].dropna(subset=['ext_cut_rip_ph', 'ext_cut_rip_selec_col'])

# 3. Realizar el Test T de Student para muestras relacionadas (Paired T-Test)
t_stat, p_value = stats.ttest_rel(df_oxidos['ext_cut_rip_ph'], df_oxidos['ext_cut_rip_selec_col'])

# 4. Mostrar e interpretar resultados
print("--- TEST T DE STUDENT (MUESTRAS RELACIONADAS) - ZONA: CUO (ÓXIDOS) ---")
print(f"Estadístico t: {t_stat:.4f}")
print(f"P-valor: {p_value:.10f}")

print("\nInterpretación:")
alpha = 0.05
if p_value < alpha:
    print("Resultado: SIGNIFICATIVO.")
    print(f"En la zona de Óxidos (CUO), existe una diferencia estadística real entre los métodos.")
    
    # Comparación de medias para ver cuál es más eficiente en Óxidos
    media_ph = df_oxidos['ext_cut_rip_ph'].mean()
    media_col = df_oxidos['ext_cut_rip_selec_col'].mean()
    
    if media_ph > media_col:
        print(f"La extracción Iso pH ({media_ph:.2f}%) es superior a la de Columna ({media_col:.2f}%).")
    else:
        print(f"La extracción por Columna ({media_col:.2f}%) es superior a la de Iso pH ({media_ph:.2f}%).")
else:
    print("Resultado: NO SIGNIFICATIVO.")
    print("En la zona CUO, no hay evidencia suficiente para afirmar que un método sea superior al otro.")

print(f"\nCantidad de muestras analizadas en CUO: {len(df_oxidos)}")

--- TEST T DE STUDENT (MUESTRAS RELACIONADAS) - ZONA: CUO (ÓXIDOS) ---
Estadístico t: 5.4024
P-valor: 0.0001594642

Interpretación:
Resultado: SIGNIFICATIVO.
En la zona de Óxidos (CUO), existe una diferencia estadística real entre los métodos.
La extracción Iso pH (76.19%) es superior a la de Columna (69.24%).

Cantidad de muestras analizadas en CUO: 13


## 2) Test de hipótesis para extracción de cobre mediante cabeza analizada

### 2.1) Test de hipotesis normalidad para óxidos de cobre metodo cabeza analizada

In [13]:
import pandas as pd
from scipy import stats
from scipy.stats import skew, kurtosis

# 1. Definición de columnas y carga de datos (Análisis de Solución)
# Incluimos 'Zona Min' para el filtrado y las nuevas variables 'anlz'
columnas_deseadas = ['Zona Min', 'ext_cut_anlz_ph', 'ext_cut_anlz_col']
excelPath = r"C:\Users\Diego\Desktop\Ultimo Esfuerzo\Copia de Base de Datos Cuprochlor Sin T Final.xlsx"

df4 = pd.read_excel(excelPath, "BD elim y reducido (3)", usecols=columnas_deseadas)

# 2. Cálculo de la variable diferencia (Análisis Solución)
df4['diferencia'] = df4['ext_cut_anlz_ph'] - df4['ext_cut_anlz_col']

# 3. Filtrado por zona mineralógica (Óxidos de Cobre - CUO)
# Limpiamos valores nulos específicos de este segmento en la nueva diferencia
datos_oxidos = df4[df4['Zona Min'] == 'CUO']['diferencia'].dropna()

# 4. Cálculo de Asimetría
asimetria = skew(datos_oxidos)
print(f"Asimetría (CUO - Solución): {asimetria:.4f}")

# 5. Cálculo de Curtosis
curto = kurtosis(datos_oxidos)
print(f"Curtosis (CUO - Solución): {curto:.4f}")

# 6. Realizar el Test de Shapiro-Wilk
# Ideal para subgrupos o zonas mineralógicas específicas
shapiro_stat, p_value = stats.shapiro(datos_oxidos)

# 7. Mostrar e interpretar resultados
print("\n--- TEST DE NORMALIDAD (SHAPIRO-WILK) - ZONA: CUO (ÓXIDOS) ---")
print(f"Estadístico W: {shapiro_stat:.4f}")
print(f"P-valor: {p_value:.4f}")

print("\nInterpretación:")
alpha = 0.05
if p_value > alpha:
    print("Resultado: Los datos de la zona CUO (solución) parecen seguir una distribución NORMAL.")
    print("Interpretación: No se rechaza la hipótesis nula (p > 0.05).")
else:
    print("Resultado: Los datos de la zona CUO (solución) NO siguen una distribución normal.")
    print("Interpretación: Se rechaza la hipótesis nula (p < 0.05).")

print(f"\nCantidad de muestras analizadas en CUO: {len(datos_oxidos)}")

Asimetría (CUO - Solución): -0.0370
Curtosis (CUO - Solución): -0.4959

--- TEST DE NORMALIDAD (SHAPIRO-WILK) - ZONA: CUO (ÓXIDOS) ---
Estadístico W: 0.9856
P-valor: 0.9964

Interpretación:
Resultado: Los datos de la zona CUO (solución) parecen seguir una distribución NORMAL.
Interpretación: No se rechaza la hipótesis nula (p > 0.05).

Cantidad de muestras analizadas en CUO: 13


### 2.2) Test t Student para la extracción de óxidos de cobre calculada mediante método de cabeza analizada

In [15]:
import pandas as pd
from scipy import stats

# 1. Definición de columnas y carga de datos (Análisis de Solución)
columnas_deseadas = ['Zona Min', 'ext_cut_anlz_ph', 'ext_cut_anlz_col']
excelPath = r"C:\Users\Diego\Desktop\Ultimo Esfuerzo\Copia de Base de Datos Cuprochlor Sin T Final.xlsx"

# Cargamos el nuevo DataFrame df4
df4 = pd.read_excel(excelPath, "BD elim y reducido (3)", usecols=columnas_deseadas)

# 2. Filtrado por zona (CUO - Óxidos de Cobre) y limpieza de nulos
# Aseguramos que el filtro use las variables de solución (anlz)
df_oxidos_anlz = df4[df4['Zona Min'] == 'CUO'].dropna(subset=['ext_cut_anlz_ph', 'ext_cut_anlz_col'])

# 3. Realizar el Test T de Student para muestras relacionadas (Paired T-Test)
t_stat, p_value = stats.ttest_rel(df_oxidos_anlz['ext_cut_anlz_ph'], df_oxidos_anlz['ext_cut_anlz_col'])

# 4. Mostrar e interpretar resultados
print("--- TEST T DE STUDENT (MUESTRAS RELACIONADAS) - ZONA: CUO (ÓXIDOS - SOLUCIÓN) ---")
print(f"Estadístico t: {t_stat:.4f}")
print(f"P-valor: {p_value:.10f}")

print("\nInterpretación:")
alpha = 0.05
if p_value < alpha:
    print("Resultado: SIGNIFICATIVO.")
    print(f"En la zona de Óxidos (CUO), existe una diferencia estadística real en la extracción analizada por solución.")
    
    # Comparación de medias dinámicas para el reporte final
    media_ph = df_oxidos_anlz['ext_cut_anlz_ph'].mean()
    media_col = df_oxidos_anlz['ext_cut_anlz_col'].mean()
    
    if media_ph > media_col:
        print(f"La extracción Iso pH ({media_ph:.2f}%) es superior a la de Columna ({media_col:.2f}%).")
    else:
        print(f"La extracción por Columna ({media_col:.2f}%) es superior a la de Iso pH ({media_ph:.2f}%).")
else:
    print("Resultado: NO SIGNIFICATIVO.")
    print("En la zona CUO (solución), no hay evidencia suficiente para afirmar que un método sea superior al otro.")

print(f"\nCantidad de muestras analizadas en CUO (Solución): {len(df_oxidos_anlz)}")

--- TEST T DE STUDENT (MUESTRAS RELACIONADAS) - ZONA: CUO (ÓXIDOS - SOLUCIÓN) ---
Estadístico t: 0.8458
P-valor: 0.4142102701

Interpretación:
Resultado: NO SIGNIFICATIVO.
En la zona CUO (solución), no hay evidencia suficiente para afirmar que un método sea superior al otro.

Cantidad de muestras analizadas en CUO (Solución): 13
